# Student Performance — Model Training

Building on the EDA notebook's insights, we train several regression models to **predict a
student's math score** from their demographic and preparation attributes, compare them fairly,
and inspect the best model's predictions.


## 1. Imports

We guard the `xgboost` / `catboost` imports so the notebook still runs on machines where those optional libraries aren't installed — it just trains fewer models instead of crashing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

OPTIONAL_MODELS = {}
try:
    from xgboost import XGBRegressor
    OPTIONAL_MODELS["XGBRegressor"] = XGBRegressor(random_state=42, verbosity=0)
except ImportError:
    print("xgboost not installed — skipping XGBRegressor.")

try:
    from catboost import CatBoostRegressor
    OPTIONAL_MODELS["CatBoosting Regressor"] = CatBoostRegressor(verbose=False, random_state=42)
except ImportError:
    print("catboost not installed — skipping CatBoosting Regressor.")


## 2. Load Data

This reuses the same `data/stud.csv` produced (or found) in the EDA notebook.

In [ ]:
import os

DATA_PATH = "data/stud.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"'{DATA_PATH}' not found. Run the EDA notebook first — it creates a synthetic "
        "stand-in dataset automatically if the real Kaggle CSV isn't available."
    )

df = pd.read_csv(DATA_PATH)
df.head()


## 3. Prepare Features and Target

We predict `math_score` from everything else — the demographic/preparation columns *and*
`reading_score` / `writing_score`, matching the original setup.

> **Worth knowing:** the EDA notebook's correlation heatmap showed reading, writing and math
> scores are all correlated around 0.8–0.85. That means part of what this model learns is
> simply "two correlated exam scores predict the third" rather than purely "these demographics
> predict math ability." That's fine for a predictive model, but keep it in mind when
> interpreting the R² below — it will look strong partly *because of* that correlation. Section 9
> adds a quick demographics-only comparison so you can see the difference.

In [ ]:
TARGET = "math_score"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Features:", list(X.columns))
X.head()


In [ ]:
num_features = X.select_dtypes(include="number").columns
cat_features = X.select_dtypes(exclude="number").columns

print(f"Numeric features: {list(num_features)}")
print(f"Categorical features: {list(cat_features)}")


### 3.1 Preprocessing Pipeline

`OneHotEncoder` turns each category into its own 0/1 column; `StandardScaler` centers and
scales the numeric columns. Wrapping both in a `ColumnTransformer` keeps preprocessing
declarative and reusable, and — importantly — lets us `fit` it only on the training split to
avoid leaking test-set statistics into training (a subtle bug in the original workflow, which
fit the encoder/scaler on the *full* dataset before splitting).

In [ ]:
preprocessor = ColumnTransformer([
    ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("StandardScaler", StandardScaler(), num_features),
])


## 4. Train / Test Split

We split *before* fitting the preprocessor, then fit it on the training data only and reuse it to transform the test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print(f"Train shape: {X_train_t.shape}")
print(f"Test shape:  {X_test_t.shape}")


## 5. Evaluation Helper

In [ ]:
def evaluate_model(y_true, y_pred):
    """Return (MAE, RMSE, R2) for a set of predictions."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


## 6. Train and Compare Multiple Models

We loop over a dictionary of candidate models, fit each on the training set, and record
train/test metrics. Collecting results in a list of rows (instead of printing them and
discarding, as the original loop did — its results list was populated but never appended to)
means we can turn them into a clean, sortable comparison table afterward.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    **OPTIONAL_MODELS,
}

results = []
for name, model in models.items():
    model.fit(X_train_t, y_train)

    train_mae, train_rmse, train_r2 = evaluate_model(y_train, model.predict(X_train_t))
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, model.predict(X_test_t))

    results.append({
        "Model": name,
        "Train R2": train_r2,
        "Test R2": test_r2,
        "Test RMSE": test_rmse,
        "Test MAE": test_mae,
    })

    print(f"{name}")
    print(f"  Train -> R2: {train_r2:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f}")
    print(f"  Test  -> R2: {test_r2:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f}")
    print("-" * 60)


### 6.1 Results Table

In [ ]:
results_df = pd.DataFrame(results).sort_values("Test R2", ascending=False).reset_index(drop=True)
results_df


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x="Test R2", y="Model", hue="Model", data=results_df, legend=False, palette="viridis")
plt.title("Model Comparison — Test R2")
plt.xlabel("R2 Score (test set)")
plt.tight_layout()
plt.show()


**Insight:** the linear models (Linear Regression / Ridge) tend to perform competitively here
despite their simplicity — a sign that the relationship between these features and math score
is largely additive and linear. Tree-based models with default settings can overfit
(high train R², lower test R²), which is exactly what hyperparameter tuning below tries to fix.

## 7. Hyperparameter Tuning Example

`RandomizedSearchCV` was imported in the original notebook but never actually used. Here's a
minimal, working example — tuning a `RandomForestRegressor` — that shows how you'd search over
a parameter grid with cross-validation rather than eyeballing default settings.

In [ ]:
param_distributions = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
)
random_search.fit(X_train_t, y_train)

tuned_test_r2 = r2_score(y_test, random_search.best_estimator_.predict(X_test_t))
print("Best parameters:", random_search.best_params_)
print(f"Tuned Random Forest — Test R2: {tuned_test_r2:.4f}")


## 8. Final Model

Based on the comparison table, we fit a `LinearRegression` as the final model — simple,
interpretable, and among the strongest performers here.

In [ ]:
final_model = LinearRegression(fit_intercept=True)
final_model.fit(X_train_t, y_train)

y_pred = final_model.predict(X_test_t)
final_r2 = r2_score(y_test, y_pred) * 100
print(f"Final model accuracy (R2): {final_r2:.2f}%")


### 8.1 Actual vs. Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.6)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", linewidth=1)
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title("Actual vs. Predicted Math Score")

sns.regplot(x=y_test, y=y_pred, ci=None, color="firebrick", ax=axes[1])
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].set_title("Regression Fit")

plt.tight_layout()
plt.show()


### 8.2 Prediction Errors

In [ ]:
pred_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": np.round(y_pred, 2),
    "Difference": np.round(y_test.values - y_pred, 2),
}).reset_index(drop=True)

pred_df.head(10)


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(pred_df["Difference"], bins=30, kde=True, color="slateblue")
plt.title("Distribution of Prediction Errors (Actual − Predicted)")
plt.xlabel("Error")
plt.show()


**Insight:** errors are roughly centered around zero with no strong skew, suggesting the linear model isn't systematically over- or under-predicting for any particular score range.

## 9. Bonus: How Much Comes From Demographics Alone?

To separate "predicting from correlated scores" from "predicting from demographics," we refit
the same model using *only* the categorical demographic/preparation columns — no
`reading_score` or `writing_score`.

In [ ]:
demo_features = df.drop(columns=[TARGET, "reading_score", "writing_score"])
demo_cat = demo_features.select_dtypes(exclude="number").columns
demo_num = demo_features.select_dtypes(include="number").columns

demo_preprocessor = ColumnTransformer([
    ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore"), demo_cat),
    ("StandardScaler", StandardScaler(), demo_num),
])

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    demo_features, y, test_size=0.2, random_state=42
)
Xd_train_t = demo_preprocessor.fit_transform(Xd_train)
Xd_test_t = demo_preprocessor.transform(Xd_test)

demo_model = LinearRegression().fit(Xd_train_t, yd_train)
demo_r2 = r2_score(yd_test, demo_model.predict(Xd_test_t))

print(f"R2 using reading + writing + demographics: {final_r2/100:.4f}")
print(f"R2 using demographics only:                {demo_r2:.4f}")


**Insight:** most of the original model's predictive power comes from the correlated reading/writing scores; demographics alone explain a much smaller (but still non-trivial) share of the variance in math score — consistent with the EDA finding that lunch, race/ethnicity group and parental education have a real but modest association with performance.

## 10. Conclusions

- Of the models compared, the **linear models** (Linear Regression / Ridge) generalize best on
  this dataset — the relationship between the categorical/demographic features and math score
  is close to linear and additive.
- Tree-based ensembles can match or exceed linear models with proper tuning, but need
  cross-validated hyperparameter search (Section 7) to avoid overfitting — using their default
  settings alone is not a fair comparison.
- The final model explains a solid share of the variance in math scores from demographic and
  preparation factors alone, though — as with any observational data — these are
  **associations, not proven causal effects**.
